# PILAR 2: ANÁLISIS DE LOS DATOS
**Trabajo Fin de Grado · Grado en Business Analytics**  
**Universidad Francisco de Vitoria · Curso 2025-26**  
**Autor: Pablo Huidobro García**

---
Este notebook implementa el segundo pilar del TFG:
- **Marco teórico:** Definición de arquitecturas de modelos
- **Métricas:** F1-Score, Precision, Recall, Accuracy
- **Entrenamiento:** ResNet50, EfficientNet-B3, ConvNeXt-Tiny
- **Comparación:** Resultados entre modelos
- **Visualización:** Matrices de confusión y curvas de aprendizaje

> **Nota:** Este notebook depende de los DataLoaders generados en el Pilar 1. Ejecutar primero `pilar1_ingenieria_dato.ipynb` o las celdas de configuración a continuación.

## 1. Configuración de rutas y entorno

In [1]:
import os
import sys
import time
import json
import warnings
import random
from pathlib import Path
warnings.filterwarnings('ignore')

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'timm', '--quiet'])

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, precision_score, recall_score, accuracy_score)

# Ruta base: raíz del TFG
# El notebook está en 03_Analisis_de_Datos_y_Modelado/notebook/
# Subimos dos niveles para llegar a la raíz TFG/
BASE_PATH = Path(os.getcwd()).resolve().parent.parent

DATASET_PATH       = str(BASE_PATH / '02_Ingenieria_de_Dato' / 'Datos brutos')
MODELOS_PATH       = str(BASE_PATH / '03_Analisis_de_Datos_y_Modelado' / 'modelos')
RESULTADOS_P2_PATH = str(BASE_PATH / '03_Analisis_de_Datos_y_Modelado' / 'resultados')

os.makedirs(MODELOS_PATH,       exist_ok=True)
os.makedirs(RESULTADOS_P2_PATH, exist_ok=True)

print(f'Base del proyecto:  {BASE_PATH}')
print(f'Dataset:            {DATASET_PATH}')
print(f'Modelos:            {MODELOS_PATH}')
print(f'Resultados Pilar 2: {RESULTADOS_P2_PATH}')

Base del proyecto:  C:\Users\Pablo\Desktop\TFG
Dataset:            C:\Users\Pablo\Desktop\TFG\02_Ingenieria_de_Dato\Datos brutos
Modelos:            C:\Users\Pablo\Desktop\TFG\03_Analisis_de_Datos_y_Modelado\modelos
Resultados Pilar 2: C:\Users\Pablo\Desktop\TFG\03_Analisis_de_Datos_y_Modelado\resultados


## 2. Configuración global

In [ ]:
# NOTA: El entrenamiento completo se ejecutó en Kaggle (GPU NVIDIA T4).
# Este notebook muestra outputs de verificación local (CPU).
# Los tiempos de latencia en resultados_completos.json corresponden a GPU T4.

CONFIG = {
    'seed':                    42,
    'image_size':              224,
    'batch_size':              32,
    'epochs':                  50,
    'early_stopping_patience': 10,
    'learning_rate':           1e-4,
    'weight_decay':            1e-5,
    'num_workers':             2,
    'augmentation_factor':     5
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(CONFIG['seed'])

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASES     = ['LATAS', 'PET', 'VIDRIO']
NUM_CLASES = len(CLASES)

print(f'Dispositivo: {DEVICE}')
print(f'PyTorch:     {torch.__version__}')

Dispositivo: cpu
PyTorch:     2.10.0+cpu


## 3. Preparación de DataLoaders

In [3]:
# ── Transforms ────────────────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),                                        # ← primero tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),         # ← luego erasing
])

transform_val_test = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── Cargar índices generados por el Pilar 1 ───────────────────────────────────
# Garantiza que ambos notebooks usan exactamente el mismo split
split_path = os.path.join(str(BASE_PATH / '02_Ingenieria_de_Dato'), 'split_indices.json')
with open(split_path) as f:
    split_indices = json.load(f)

train_idx = split_indices['train']
val_idx   = split_indices['val']
test_idx  = split_indices['test']
n         = len(train_idx) + len(val_idx) + len(test_idx)

# ── Tres ImageFolder independientes → sin data leakage ────────────────────────
dataset_train = ImageFolder(root=DATASET_PATH, transform=transform_train)
dataset_val   = ImageFolder(root=DATASET_PATH, transform=transform_val_test)
dataset_test  = ImageFolder(root=DATASET_PATH, transform=transform_val_test)

train_dataset = torch.utils.data.Subset(dataset_train, train_idx)
val_dataset   = torch.utils.data.Subset(dataset_val,   val_idx)
test_dataset  = torch.utils.data.Subset(dataset_test,  test_idx)

# ── DataLoaders ────────────────────────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=CONFIG['num_workers'])
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=CONFIG['num_workers'])
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=CONFIG['num_workers'])

print(f'Training:   {len(train_dataset):5d} imágenes')
print(f'Validation: {len(val_dataset):5d} imágenes')
print(f'Test:       {len(test_dataset):5d} imágenes')
print(f'Split:      {len(train_dataset)/n*100:.0f}% / '
      f'{len(val_dataset)/n*100:.0f}% / '
      f'{len(test_dataset)/n*100:.0f}%')

Training:    1584 imágenes
Validation:   339 imágenes
Test:         341 imágenes
Split:      70% / 15% / 15%


---
## PILAR 2: ANÁLISIS DE LOS DATOS
---
### 2.1 Marco Teórico: Arquitecturas de modelos

Se implementan tres arquitecturas de Deep Learning con Transfer Learning desde ImageNet:

| Modelo | Descripción | Ventaja |
|---|---|---|
| **ResNet50** | Red residual de 50 capas | Resuelve vanishing gradient |
| **EfficientNet-B3** | Escalado compuesto optimizado | Mejor ratio precisión/parámetros |
| **ConvNeXt-Tiny** | CNN modernizada inspirada en ViT | Combina CNN y Transformers |

### 2.2 Definición del modelo base

In [4]:
class ModeloClasificador(nn.Module):
    """Clasificador con Transfer Learning intercambiable por arquitectura."""
    def __init__(self, nombre_modelo, num_clases, pretrained=True):
        super(ModeloClasificador, self).__init__()
        self.nombre   = nombre_modelo
        self.backbone = timm.create_model(nombre_modelo, pretrained=pretrained)

        if hasattr(self.backbone, 'fc'):
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
        elif hasattr(self.backbone, 'classifier'):
            num_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        elif hasattr(self.backbone, 'head'):
            if hasattr(self.backbone.head, 'fc'):
                num_features = self.backbone.head.fc.in_features
                self.backbone.head.fc = nn.Identity()
            else:
                num_features = self.backbone.head.in_features
                self.backbone.head = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, num_clases)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

print('Clase ModeloClasificador definida correctamente')

Clase ModeloClasificador definida correctamente


### 2.3 Definición de métricas de evaluación

In [5]:
def calcular_metricas(modelo, dataloader, device):
    """
    Calcula F1, Precision, Recall, Accuracy y tiempo de inferencia.
    Métrica principal: F1-Score weighted (balance precision/recall crítico en SDDR).
    """
    modelo.eval()
    todas_predicciones = []
    todas_etiquetas    = []
    tiempos_inferencia = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Sincronizar GPU antes de medir para obtener tiempos reales
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_time = time.time()

            outputs = modelo(inputs)

            if device.type == 'cuda':
                torch.cuda.synchronize()
            tiempo_batch = (time.time() - start_time) * 1000

            tiempos_inferencia.append(tiempo_batch / len(inputs))
            _, predicciones = torch.max(outputs, 1)
            todas_predicciones.extend(predicciones.cpu().numpy())
            todas_etiquetas.extend(labels.cpu().numpy())

    metricas = {
        'f1':                   f1_score(todas_etiquetas, todas_predicciones, average='weighted'),
        'precision':            precision_score(todas_etiquetas, todas_predicciones, average='weighted'),
        'recall':               recall_score(todas_etiquetas, todas_predicciones, average='weighted'),
        'accuracy':             accuracy_score(todas_etiquetas, todas_predicciones),
        'tiempo_inferencia_ms': np.mean(tiempos_inferencia)
    }
    return metricas, todas_predicciones, todas_etiquetas

print('Función calcular_metricas definida')
print('Métricas: F1-Score (principal), Precision, Recall, Accuracy, Tiempo de inferencia')

Función calcular_metricas definida
Métricas: F1-Score (principal), Precision, Recall, Accuracy, Tiempo de inferencia


### 2.4 Función de entrenamiento

In [6]:
def entrenar_modelo(modelo, train_loader, val_loader, config, device, nombre_archivo):
    """Entrena el modelo con early stopping basado en F1-Score de validación."""
    criterio    = nn.CrossEntropyLoss()
    optimizador = optim.AdamW(modelo.parameters(),
                              lr=config['learning_rate'],
                              weight_decay=config['weight_decay'])
    scheduler   = optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode='max',
                                                        factor=0.5, patience=5)

    mejor_f1          = 0
    epocas_sin_mejora = 0
    historial = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_accuracy': []}

    for epoca in range(config['epochs']):
        # Entrenamiento
        modelo.train()
        loss_train = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizador.zero_grad()
            loss = criterio(modelo(inputs), labels)
            loss.backward()
            optimizador.step()
            loss_train += loss.item()

        # Validación
        modelo.eval()
        loss_val            = 0.0
        todas_predicciones  = []
        todas_etiquetas     = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = modelo(inputs)
                loss_val += criterio(outputs, labels).item()
                _, preds = torch.max(outputs, 1)
                todas_predicciones.extend(preds.cpu().numpy())
                todas_etiquetas.extend(labels.cpu().numpy())

        f1_val       = f1_score(todas_etiquetas, todas_predicciones, average='weighted')
        accuracy_val = accuracy_score(todas_etiquetas, todas_predicciones)

        scheduler.step(f1_val)
        historial['train_loss'].append(loss_train / len(train_loader))
        historial['val_loss'].append(loss_val / len(val_loader))
        historial['val_f1'].append(f1_val)
        historial['val_accuracy'].append(accuracy_val)

        if f1_val > mejor_f1:
            mejor_f1          = f1_val
            epocas_sin_mejora = 0
            torch.save(modelo.state_dict(),
                       os.path.join(MODELOS_PATH, f'{nombre_archivo}_best.pth'))
        else:
            epocas_sin_mejora += 1

        if (epoca + 1) % 5 == 0:
            print(f'  Época {epoca+1:3d} | '
                  f'Train Loss: {historial["train_loss"][-1]:.4f} | '
                  f'Val Loss: {historial["val_loss"][-1]:.4f} | '
                  f'Val F1: {f1_val:.4f} | '
                  f'Val Acc: {accuracy_val:.4f}')

        if epocas_sin_mejora >= config['early_stopping_patience']:
            print(f'  Early stopping en época {epoca+1}')
            break

    modelo.load_state_dict(torch.load(os.path.join(MODELOS_PATH, f'{nombre_archivo}_best.pth')))
    print(f'  Mejor F1-Score: {mejor_f1:.4f}')
    return modelo, historial

print('Función entrenar_modelo definida')

Función entrenar_modelo definida


### 2.5 Entrenamiento de los tres modelos

In [7]:
modelos_configuracion = [
    ('resnet50',        'ResNet50',       'ResNet50'),
    ('efficientnet_b3', 'EfficientNet-B3','EfficientNet_B3'),
    ('convnext_tiny',   'ConvNeXt-Tiny',  'ConvNeXt_Tiny')
]

resultados_modelos = {}

for nombre_modelo, nombre_display, nombre_archivo in modelos_configuracion:
    print(f'\n{"-" * 60}')
    print(f'ENTRENANDO: {nombre_display}')
    print(f'{"-" * 60}')

    modelo = ModeloClasificador(nombre_modelo, NUM_CLASES, pretrained=True).to(DEVICE)
    modelo_entrenado, historial = entrenar_modelo(
        modelo, train_loader, val_loader, CONFIG, DEVICE, nombre_archivo
    )

    # Guardar historial
    with open(os.path.join(MODELOS_PATH, f'historia_{nombre_archivo}.json'), 'w') as f:
        json.dump(historial, f, indent=2)

    # Evaluar en test
    metricas_test, predicciones_test, etiquetas_test = calcular_metricas(
        modelo_entrenado, test_loader, DEVICE
    )

    print(f'\n  Resultados en test:')
    print(f'    F1-Score:   {metricas_test["f1"]:.4f}')
    print(f'    Precision:  {metricas_test["precision"]:.4f}')
    print(f'    Recall:     {metricas_test["recall"]:.4f}')
    print(f'    Accuracy:   {metricas_test["accuracy"]:.4f}')
    print(f'    Tiempo/img: {metricas_test["tiempo_inferencia_ms"]:.2f} ms')

    resultados_modelos[nombre_modelo] = {
        'nombre_display': nombre_display,
        'nombre_archivo': nombre_archivo,
        'historial':      historial,
        'metricas_test':  metricas_test,
        'predicciones':   predicciones_test,
        'etiquetas':      etiquetas_test
    }


------------------------------------------------------------
ENTRENANDO: ResNet50
------------------------------------------------------------


KeyboardInterrupt: 

### 2.6 Comparación de resultados entre modelos

In [ ]:
df_comparacion = pd.DataFrame({
    'Modelo':    [resultados_modelos[m]['nombre_display'] for m in resultados_modelos],
    'F1-Score':  [resultados_modelos[m]['metricas_test']['f1'] for m in resultados_modelos],
    'Precision': [resultados_modelos[m]['metricas_test']['precision'] for m in resultados_modelos],
    'Recall':    [resultados_modelos[m]['metricas_test']['recall'] for m in resultados_modelos],
    'Accuracy':  [resultados_modelos[m]['metricas_test']['accuracy'] for m in resultados_modelos],
    'Tiempo_ms': [resultados_modelos[m]['metricas_test']['tiempo_inferencia_ms'] for m in resultados_modelos]
})

print(df_comparacion.to_string(index=False))

# Guardar CSV
df_comparacion.to_csv(os.path.join(RESULTADOS_P2_PATH, 'comparativa_modelos.csv'), index=False)
print('\nGuardado: 03_Analisis_de_Datos_y_Modelado/resultados/comparativa_modelos.csv')

mejor_modelo_nombre = df_comparacion.loc[df_comparacion['F1-Score'].idxmax(), 'Modelo']
print(f'\nModelo óptimo: {mejor_modelo_nombre}')

# Guardar resultados completos
resultados_completos = {
    m: {
        'nombre_display': resultados_modelos[m]['nombre_display'],
        'metricas_test':  resultados_modelos[m]['metricas_test']
    }
    for m in resultados_modelos
}
with open(os.path.join(RESULTADOS_P2_PATH, 'resultados_completos.json'), 'w', encoding='utf-8') as f:
    json.dump(resultados_completos, f, indent=2, ensure_ascii=False)
print('Guardado: 03_Analisis_de_Datos_y_Modelado/resultados/resultados_completos.json')

### 2.7 Visualización: Comparativa final de métricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colores = ['#E74C3C', '#3498DB', '#2ECC71']

for ax, metrica, titulo in zip(
    axes.flat,
    ['F1-Score', 'Precision', 'Recall', 'Tiempo_ms'],
    ['F1-Score', 'Precision', 'Recall', 'Tiempo de Inferencia (ms)']
):
    ax.bar(df_comparacion['Modelo'], df_comparacion[metrica], color=colores)
    ax.set_title(f'{titulo} por Modelo', fontsize=12, fontweight='bold')
    ax.set_ylabel(titulo)
    if metrica != 'Tiempo_ms':
        ax.set_ylim([0.7, 1.0])
    for i, v in enumerate(df_comparacion[metrica]):
        label = f'{v:.4f}' if metrica != 'Tiempo_ms' else f'{v:.2f}ms'
        ax.text(i, v + (0.01 if metrica != 'Tiempo_ms' else 0.5),
                label, ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(RESULTADOS_P2_PATH, 'comparativa_final.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: 03_Analisis_de_Datos_y_Modelado/resultados/comparativa_final.png')

### 2.8 Visualización: Matrices de confusión

In [ ]:
for nombre_modelo in resultados_modelos:
    resultado      = resultados_modelos[nombre_modelo]
    nombre_archivo = resultado['nombre_archivo']
    cm = confusion_matrix(resultado['etiquetas'], resultado['predicciones'])

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASES, yticklabels=CLASES)
    ax.set_title(f'{resultado["nombre_display"]}\nF1={resultado["metricas_test"]["f1"]:.4f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Valor Real')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_P2_PATH, f'confusion_{nombre_archivo}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Guardado: confusion_{nombre_archivo}.png')

### 2.9 Visualización: Curvas de aprendizaje

In [ ]:
for nombre_modelo in resultados_modelos:
    resultado      = resultados_modelos[nombre_modelo]
    nombre_archivo = resultado['nombre_archivo']
    historial      = resultado['historial']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(historial['train_loss'], label='Train Loss', linewidth=2)
    axes[0].plot(historial['val_loss'],   label='Validation Loss', linewidth=2)
    axes[0].set_title(f'Curvas de Loss - {resultado["nombre_display"]}',
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(historial['val_f1'], label='Validation F1-Score',
                 linewidth=2, color='green')
    axes[1].set_title(f'Evolución F1-Score - {resultado["nombre_display"]}',
                      fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('F1-Score')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_P2_PATH, f'curvas_{nombre_archivo}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Guardado: curvas_{nombre_archivo}.png')

---
## Resumen Pilar 2

**Outputs generados:**
- `03_Analisis_de_Datos_y_Modelado/modelos/ResNet50_best.pth`
- `03_Analisis_de_Datos_y_Modelado/modelos/EfficientNet_B3_best.pth`
- `03_Analisis_de_Datos_y_Modelado/modelos/ConvNeXt_Tiny_best.pth`
- `03_Analisis_de_Datos_y_Modelado/modelos/historia_*.json` (x3)
- `03_Analisis_de_Datos_y_Modelado/resultados/comparativa_modelos.csv`
- `03_Analisis_de_Datos_y_Modelado/resultados/comparativa_final.png`
- `03_Analisis_de_Datos_y_Modelado/resultados/confusion_*.png` (x3)
- `03_Analisis_de_Datos_y_Modelado/resultados/curvas_*.png` (x3)
- `03_Analisis_de_Datos_y_Modelado/resultados/resultados_completos.json`